In [ ]:
import re
import json
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

# Configuração básica do Dataset

1.   Deefinição de caminhos
2.   Validação de possíveis sujeiras (Em um futuro usar o Regex para isso)
1.   Função para agregar alguns registros de abas





In [ ]:
EXCEL_PATH = "2021_coleta.xlsx"
OUTPUT_DIR = Path("saida_analise")
OUTPUT_DIR.mkdir(exist_ok=True)


MISSING_TOKENS = {
    "", " ", "  ", "-", "--", "---", "na", "n/a", "null", "none",
    "não informado", "nao informado", "não se aplica", "nao se aplica",
    "ignorado", "sem informação", "sem informacao", "nan"
}


BOOLEAN_TRUE = {"sim", "s", "yes", "y", "true", "1", "x"}
BOOLEAN_FALSE = {"nao", "não", "n", "no", "false", "0"}


AGG_CONFIG = {
    "Projetos de Pesquisa": {
        "group_keys": [
            "Código do PPG", "Nome do PPG", "Nome do Projeto de Pesquisa",
            "Linha de Pesquisa", "Área de Concentração",
            "Natureza do Projeto de Pesquisa", "Situação", "Data de Início"
        ],
        "list_columns": [
            "Nome Membro do Projeto", "Categoria Membro do Projeto",
            "Membro Responsável", "Nome do Financiador",
            "Documento do Financiador", "Financiador Estrangeiro (Sim/Não)"
        ],
        "date_min_columns": ["Data Início Membro", "Data Início Financiador"],
        "date_max_columns": ["Data Fim Membro", "Data Fim Financiador"]
    },
    "Trabalhos de conclusão": {
        "group_keys": [
            "Código do PPG", "Nome do PPG", "Nome Trabalho de Conclusão",
            "Nome Autor", "Tipo do Trabalho de Conclusão", "Data da Defesa"
        ],
        "list_columns": [
            "Nome Orientador", "Nome Membro da Banca",
            "Categoria Membro da Banca", "Nome do Financiador"
        ],
        "date_min_columns": [],
        "date_max_columns": []
    }
}


# Funções para análise

| Nome da Função | Descrição |
| :--- | :--- |
| `normalize_text` | Limpa o texto removendo acentos, colapsando espaços múltiplos e convertendo para minúsculas. |
| `snake_case` | Formata strings para o padrão `snake_case`, removendo caracteres especiais e substituindo espaços por sublinhados. |
| `is_probably_date` | Verifica se uma Series provavelmente contém datas com base em um limite de conversão bem-sucedida. |
| `is_probably_numeric` | Verifica se uma Series provavelmente contém valores numéricos ao tentar converter strings em números. |
| `is_probably_boolean` | Determina se uma Series contém textos do tipo booleano (ex: "True", "False", "Sim", "Não"). |
| `describe_column_type` | Analisa uma coluna para retornar seu tipo lógico, contagem de nulos e amostras de valores únicos. |
| `standardize_yes_no` | Padroniza diversas entradas textuais de "Sim/Não" para um formato padrão "Sim" ou "Não". |
| `coerce_numeric` | Converte uma Series para tipos numéricos, tratando separadores de vírgula decimal e erros. |
| `coerce_datetime` | Converte uma Series em objetos datetime, priorizando o formato com o dia primeiro. |
| `unique_join` | Une valores únicos e não nulos de uma lista em uma única string separada por um delimitador. |

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)

    text_ascii = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    text_lower = text_ascii.lower().strip()

    if text_lower in MISSING_TOKENS:
        return np.nan

    return text


def snake_case(text):
    if text is None:
        return text
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("utf-8")
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def is_probably_date(series, threshold=0.7):
    if series.dropna().empty:
        return False
    parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
    ratio = parsed.notna().mean()
    return ratio >= threshold


def is_probably_numeric(series, threshold=0.8):
    if series.dropna().empty:
        return False
    coerced = pd.to_numeric(series.astype(str).str.replace(",", ".", regex=False), errors="coerce")
    ratio = coerced.notna().mean()
    return ratio >= threshold


def is_probably_boolean(series, threshold=0.9):
    if series.dropna().empty:
        return False
    vals = (
        series.dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8"))
    )
    ratio = vals.isin(BOOLEAN_TRUE.union(BOOLEAN_FALSE)).mean()
    return ratio >= threshold


def describe_column_type(series):
    s = series.copy()

    result = {
        "dtype_pandas": str(s.dtype),
        "tipo_logico": None,
        "n_total": int(len(s)),
        "n_na": int(s.isna().sum()),
        "pct_na": round(float(s.isna().mean() * 100), 2) if len(s) else 0.0,
        "n_unicos": int(s.nunique(dropna=True)),
        "amostra_valores": s.dropna().astype(str).head(5).tolist()
    }

    if pd.api.types.is_datetime64_any_dtype(s):
        result["tipo_logico"] = "datetime"
    elif pd.api.types.is_numeric_dtype(s):
        if pd.api.types.is_integer_dtype(s):
            result["tipo_logico"] = "inteiro"
        else:
            result["tipo_logico"] = "numerico_continuo"
    else:
        if is_probably_boolean(s):
            result["tipo_logico"] = "booleano_textual"
        elif is_probably_date(s):
            result["tipo_logico"] = "data_textual"
        elif is_probably_numeric(s):
            result["tipo_logico"] = "numerico_textual"
        else:
            nunique = s.nunique(dropna=True)
            if nunique <= max(20, len(s) * 0.05):
                result["tipo_logico"] = "categorica_textual"
            else:
                result["tipo_logico"] = "texto_livre"
    return result


def standardize_yes_no(series):
    def _map(v):
        if pd.isna(v):
            return np.nan
        x = unicodedata.normalize("NFKD", str(v).strip().lower()).encode("ascii", "ignore").decode("utf-8")
        if x in BOOLEAN_TRUE:
            return "Sim"
        if x in BOOLEAN_FALSE:
            return "Não"
        return str(v).strip()
    return series.map(_map)


def coerce_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(",", ".", regex=False), errors="coerce")


def coerce_datetime(series):
    return pd.to_datetime(series, errors="coerce", dayfirst=True)


def unique_join(values, sep=" | "):
    vals = []
    seen = set()
    for v in values:
        if pd.isna(v):
            continue
        vv = str(v).strip()
        if not vv:
            continue
        if vv not in seen:
            vals.append(vv)
            seen.add(vv)
    return sep.join(vals) if vals else np.nan


# Leitura do excel e retorno do dicionário


In [ ]:
def load_excel_sheets(excel_path):
    xls = pd.ExcelFile(excel_path)
    sheets = {sheet_name: pd.read_excel(excel_path, sheet_name=sheet_name) for sheet_name in xls.sheet_names}
    return sheets

sheets = load_excel_sheets(EXCEL_PATH)
list(sheets.keys())

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

['Programa',
 'Proposta',
 'Financiadores',
 'Linhas de Pesquisa',
 'Projetos de Pesquisa',
 'Disciplinas',
 'Turmas',
 'Docentes',
 'Discentes',
 'Participantes Externos',
 'Trabalhos de Conclusão',
 'Produção Intelectual',
 'Produções Mais Relevantes',
 'Pós-Doc',
 'Projeto de Cooperação entre Ins',
 'Egresso']

# Baixa padronização


1. Remover colunas vazias
2. Limpar espaços em branco nas colunas de texto
1. Substituir os marcadores de posição por NaN
2. Alterar os atributos para snake_case





In [ ]:
def standardize_dataframe(df):

    out = df.copy()

    # Apaga colunas Vazias
    out = out.dropna(axis=1, how="all")

    # Normalização textual
    for col in out.columns:
        if out[col].dtype == "object":
            out[col] = out[col].map(normalize_text)

    return out

standardized_sheets = {name: standardize_dataframe(df) for name, df in sheets.items()}

# Dataset Resumo for TAB

In [ ]:
def summarize_dataset(sheets_dict):
    rows = []
    for sheet_name, df in sheets_dict.items():
        rows.append({
            "aba": sheet_name,
            "linhas": df.shape[0],
            "colunas": df.shape[1],
            "celulas": df.shape[0] * df.shape[1],
            "colunas_totalmente_vazias": int(df.isna().all().sum()),
            "linhas_totalmente_vazias": int(df.isna().all(axis=1).sum()),
            "percentual_ausencia_global": round(float(df.isna().mean().mean() * 100), 2)
        })
    return pd.DataFrame(rows).sort_values(["linhas", "colunas"], ascending=False).reset_index(drop=True)

dataset_summary = summarize_dataset(standardized_sheets)
dataset_summary

,aba,linhas,colunas,celulas,colunas_totalmente_vazias,linhas_totalmente_vazias,percentual_ausencia_global
0,Produção Intelectual,1801,21,37821,0,0,16.32
1,Projetos de Pesquisa,122,24,2928,0,0,23.19
2,Participantes Externos,57,13,741,0,0,7.29
3,Discentes,44,18,792,0,0,8.08
4,Disciplinas,38,13,494,0,0,0.00
5,Trabalhos de Conclusão,30,17,510,0,0,11.76
6,Financiadores,30,13,390,0,0,0.00
7,Egresso,19,11,209,0,0,0.00
8,Docentes,18,15,270,0,0,5.93
9,Turmas,12,17,204,0,0,0.00


# Contruir dicionário de dados


In [ ]:
def build_data_dictionary(sheets_dict):
    records = []
    for sheet_name, df in sheets_dict.items():
        for col in df.columns:
            meta = describe_column_type(df[col])
            records.append({
                "aba": sheet_name,
                "coluna": col,
                "coluna_snake_case": snake_case(col),
                **meta
            })
    return pd.DataFrame(records)

data_dictionary = build_data_dictionary(standardized_sheets)
data_dictionary.head(40)

/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dat

,aba,coluna,coluna_snake_case,dtype_pandas,tipo_logico,n_total,n_na,pct_na,n_unicos,amostra_valores
0,Programa,Calendário,calendario,object,categorica_textual,1,0,0.0,1,[Coleta de Informações 2021]
1,Programa,Ano do Calendário,ano_do_calendario,int64,inteiro,1,0,0.0,1,[2021]
2,Programa,Data-Hora do Envio,data_hora_do_envio,object,data_textual,1,0,0.0,1,[30/03/2023 - 15:07]
3,Programa,Código do PPG,codigo_do_ppg,object,categorica_textual,1,0,0.0,1,[22001018081P2]
4,Programa,Nome do PPG,nome_do_ppg,object,categorica_textual,1,0,0.0,1,[Engenharia Elétrica e de Computação]
5,Programa,Área de Avaliação,area_de_avaliacao,object,categorica_textual,1,0,0.0,1,[ENGENHARIAS IV]
6,Programa,Área Básica,area_basica,object,categorica_textual,1,0,0.0,1,[ENGENHARIA ELÉTRICA]
7,Programa,Regime Letivo,regime_letivo,object,categorica_textual,1,0,0.0,1,[SEMESTRAL]
8,Programa,IES Sigla,ies_sigla,object,categorica_textual,1,0,0.0,1,[UFC]
9,Programa,IES Nome,ies_nome,object,categorica_textual,1,0,0.0,1,[UNIVERSIDADE FEDERAL DO CEARÁ]


# Lista de sujeiras

In [ ]:
def detect_dirty_patterns(df):
    findings = []


    if df.empty or len(df.columns) == 0:
        return pd.DataFrame(columns=[
            "coluna", "nulos", "pct_nulos", "duplicados_na_coluna",
            "espacos_inicio_fim", "multiplos_espacos", "placeholders_ausencia",
            "datas_invalidas", "valores_unicos", "coluna_constante"
        ])

    for col in df.columns:
        s = df[col]
        total_rows = len(s)

        if s.dtype == "object":
            raw = s.astype(str)
            has_leading_trailing_spaces = int((raw != raw.str.strip()).sum())
            multiple_internal_spaces = int(raw.str.contains(r"\s{2,}", regex=True, na=False).sum())

            empty_like = int(
                s.fillna("")
                 .astype(str)
                 .str.strip()
                 .str.lower()
                 .map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8"))
                 .isin(MISSING_TOKENS)
                 .sum()
            )
        else:
            has_leading_trailing_spaces = 0
            multiple_internal_spaces = 0
            empty_like = 0

        invalid_date_count = 0
        if is_probably_date(s):
            parsed_dates = pd.to_datetime(s, errors="coerce", dayfirst=True)
            invalid_date_count = int(parsed_dates.isna().sum() - s.isna().sum())
            invalid_date_count = max(0, invalid_date_count)

        n_nulos = int(s.isna().sum())
        pct_nulos = round(float((n_nulos / total_rows) * 100), 2) if total_rows > 0 else 0.0
        n_unicos = int(s.nunique(dropna=True))

        findings.append({
            "coluna": col,
            "nulos": n_nulos,
            "pct_nulos": pct_nulos,
            "duplicados_na_coluna": int(s.duplicated(keep=False).sum()) if total_rows > 0 else 0,
            "espacos_inicio_fim": has_leading_trailing_spaces,
            "multiplos_espacos": multiple_internal_spaces,
            "placeholders_ausencia": empty_like,
            "datas_invalidas": invalid_date_count,
            "valores_unicos": n_unicos,
            "coluna_constante": bool(n_unicos <= 1 and total_rows > 0)
        })

    return pd.DataFrame(findings).sort_values(
        ["pct_nulos", "placeholders_ausencia"],
        ascending=False
    ).reset_index(drop=True)

In [ ]:
dirty_reports = {name: detect_dirty_patterns(df) for name, df in standardized_sheets.items()}
dirty_reports[list(dirty_reports.keys())[4]].head(20)

/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dat

,coluna,nulos,pct_nulos,duplicados_na_coluna,espacos_inicio_fim,multiplos_espacos,placeholders_ausencia,datas_invalidas,valores_unicos,coluna_constante
0,Data Fim Financiador,116,95.08,116,0,0,116,0,6,False
1,Nome do Financiador,113,92.62,120,0,0,113,0,4,False
2,Financiador Estrangeiro (Sim/Não),113,92.62,122,0,0,113,0,1,True
3,Data Início Financiador,113,92.62,115,0,0,113,0,8,False
4,Documento do Financiador,113,92.62,120,0,0,0,0,4,False
5,Data Fim Membro,75,61.48,105,0,0,75,0,25,False
6,Nome Membro do Projeto,9,7.38,75,0,0,9,0,66,False
7,Categoria Membro do Projeto,9,7.38,122,0,0,9,0,5,False
8,Membro Responsável,9,7.38,122,0,0,9,0,2,False
9,Data Início Membro,9,7.38,101,0,0,9,0,39,False


# Cnversão inteligente de **tipos**



```
meta = describe_column_type(s)
logical_type = meta["tipo_logico"]
```
Ele classifica como:


*   data_textual
*   numerico_textual
*   booleano_textual
*   categorica_textual







In [ ]:
def infer_and_cast_types(df):
    out = df.copy()
    cast_log = []

    for col in out.columns:
        s = out[col]
        meta = describe_column_type(s)
        logical_type = meta["tipo_logico"]

        original_dtype = str(s.dtype)
        new_dtype = original_dtype

        try:
            if logical_type == "data_textual":
                out[col] = coerce_datetime(s)
                new_dtype = str(out[col].dtype)

            elif logical_type == "numerico_textual":
                numeric = coerce_numeric(s)

                # tenta inteiro pandas nullable quando apropriado
                if numeric.dropna().apply(float.is_integer).all() if not numeric.dropna().empty else False:
                    out[col] = numeric.astype("Int64")
                else:
                    out[col] = numeric.astype("Float64")
                new_dtype = str(out[col].dtype)

            elif logical_type == "booleano_textual":
                out[col] = standardize_yes_no(s).astype("category")
                new_dtype = str(out[col].dtype)

            elif logical_type == "categorica_textual":
                out[col] = s.astype("category")
                new_dtype = str(out[col].dtype)

        except Exception as e:
            cast_log.append({
                "coluna": col,
                "tipo_logico_detectado": logical_type,
                "dtype_original": original_dtype,
                "dtype_novo": new_dtype,
                "status": f"erro: {e}"
            })
            continue

        cast_log.append({
            "coluna": col,
            "tipo_logico_detectado": logical_type,
            "dtype_original": original_dtype,
            "dtype_novo": new_dtype,
            "status": "ok"
        })

    return out, pd.DataFrame(cast_log)

typed_sheets = {}
cast_logs = {}

for name, df in standardized_sheets.items():
    typed_sheets[name], cast_logs[name] = infer_and_cast_types(df)

cast_logs[list(cast_logs.keys())[0]].head(20)

/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
/tmp/ipykernel_604/1125811358.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dat

,coluna,tipo_logico_detectado,dtype_original,dtype_novo,status
0,Calendário,categorica_textual,object,category,ok
1,Ano do Calendário,inteiro,int64,int64,ok
2,Data-Hora do Envio,data_textual,object,datetime64[ns],ok
3,Código do PPG,categorica_textual,object,category,ok
4,Nome do PPG,categorica_textual,object,category,ok
5,Área de Avaliação,categorica_textual,object,category,ok
6,Área Básica,categorica_textual,object,category,ok
7,Regime Letivo,categorica_textual,object,category,ok
8,IES Sigla,categorica_textual,object,category,ok
9,IES Nome,categorica_textual,object,category,ok


# Tratamento de dados faltantes


*   (qtd creditos, qtd meses) --> 0
*   (tempo medio de projeto, duracao vinculo, indicadores) --> média
*   (Tempo de titulação, Duração de projetos, Tempo de orientação) --> mediana
*   (Área de Avaliação, Nível do Curso, Situação, Categoria, Tipo da Produção, Categoria do Docente, Categoria do Discente) --> MODA (consultar professor)
*   (Nome do PPG, Nome Discente, Nome Docente, Nome do Projeto de Pesquisa, Nome Trabalho de Conclusão, Título da Produção) --> Sem isso a linha não serve






In [ ]:
def treat_missing_values(
    df,
    zero_fill_cols=None,
    mean_fill_cols=None,
    median_fill_cols=None,
    mode_fill_cols=None,
    drop_rows_if_missing_in=None
):
    """
    Tratamento configurável de ausências.
    """
    out = df.copy()

    zero_fill_cols = zero_fill_cols or []
    mean_fill_cols = mean_fill_cols or []
    median_fill_cols = median_fill_cols or []
    mode_fill_cols = mode_fill_cols or []
    drop_rows_if_missing_in = drop_rows_if_missing_in or []

    for col in zero_fill_cols:
        if col in out.columns:
            out[col] = out[col].fillna(0)

    for col in mean_fill_cols:
        if col in out.columns and pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].fillna(out[col].mean())

    for col in median_fill_cols:
        if col in out.columns and pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].fillna(out[col].median())

    for col in mode_fill_cols:
        if col in out.columns:
            mode = out[col].mode(dropna=True)
            if len(mode):
                out[col] = out[col].fillna(mode.iloc[0])

    if drop_rows_if_missing_in:
        valid_cols = [c for c in drop_rows_if_missing_in if c in out.columns]
        if valid_cols:
            out = out.dropna(subset=valid_cols)

    return out


# Funções de duplicidade




```
# prod_total = remove_duplicates_for_total(df_producao, ["Título da Produção", "Ano da Produção"])
# coautoria = identify_coauthorship(df_producao, ["Título da Produção", "Ano da Produção"])
```





In [ ]:
def remove_duplicates_for_total(df, subset_cols):
    """
    Exemplo para produção total:
    conta apenas uma vez a mesma entidade.
    """
    valid_subset = [c for c in subset_cols if c in df.columns]
    if not valid_subset:
        return df.copy()
    return df.drop_duplicates(subset=valid_subset, keep="first").copy()


def identify_coauthorship(df, subset_cols):
    """
    Exemplo para coautoria:
    identifica quantas vezes a mesma produção aparece.
    """
    valid_subset = [c for c in subset_cols if c in df.columns]
    if not valid_subset:
        return pd.DataFrame()

    return (
        df.groupby(valid_subset, dropna=False)
          .size()
          .reset_index(name="n_participacoes")
          .sort_values("n_participacoes", ascending=False)
    )


# Agrupamento de entidades
- Exemplo para linhas da mesma entidade na base (consolidando participantes, financiadores, banca)

In [ ]:
def aggregate_entity_table(df, group_keys, list_columns=None, date_min_columns=None, date_max_columns=None):
    """
    Agrupa linhas que representam a mesma entidade, consolidando participantes,
    financiadores, banca etc.
    """
    if df.empty:
        return df.copy()

    # Filtra colunas existentes
    list_columns = [c for c in (list_columns or []) if c in df.columns]
    date_min_columns = [c for c in (date_min_columns or []) if c in df.columns]
    date_max_columns = [c for c in (date_max_columns or []) if c in df.columns]
    group_keys = [c for c in group_keys if c in df.columns]

    if not group_keys:
        raise ValueError("Nenhuma group_key válida encontrada no DataFrame.")

    working = df.copy()

    for c in date_min_columns + date_max_columns:
        if not pd.api.types.is_datetime64_any_dtype(working[c]):
            working[c] = pd.to_datetime(working[c], errors="coerce", dayfirst=True)

    agg_spec = {}

    # 1. Agregação de listas (Custom unique_join)
    for c in list_columns:
        agg_spec[c] = unique_join

    # 2. Agregação de datas (Min/Max)
    for c in date_min_columns:
        agg_spec[c] = "min"

    for c in date_max_columns:
        agg_spec[c] = "max"

    # 3. Tratamento das colunas restantes
    remaining_cols = [c for c in working.columns if c not in group_keys and c not in agg_spec]
    for c in remaining_cols:
        agg_spec[c] = "first"

    # Executa a agregação
    result = (
        working.groupby(group_keys, dropna=False, as_index=False)
               .agg(agg_spec)
    )

    return result

# Padronização em algumas colunas recorrentes

In [ ]:
def standardize_domain_columns(df):

    out = df.copy()

    for col in out.columns:
        col_norm = snake_case(col)

        # 1. Padronização Sim/Não
        if any(token in col_norm for token in ["estrangeiro", "responsavel", "principal", "participou"]):
            out[col] = standardize_yes_no(out[col])

        # 2. Padronização de Datas
        if any(token in col_norm for token in ["data", "inicio", "fim", "periodo"]):
            # Só tenta converter se não for datetime e se "parecer" uma data
            if not pd.api.types.is_datetime64_any_dtype(out[col]) and is_probably_date(out[col], threshold=0.5):
                out[col] = coerce_datetime(out[col])

        # 3. Padronização de Ano
        if col_norm.startswith("ano") or col_norm.endswith("_ano"):
            numeric = coerce_numeric(out[col])
            if not numeric.dropna().empty:
                # Compara o valor com sua versão arredondada para ver se é inteiro
                if (numeric.dropna() % 1 == 0).all():
                    out[col] = numeric.astype("Int64")

    return out
final_sheets = {name: standardize_domain_columns(df) for name, df in typed_sheets.items()}

# Geração de relatório de tipos e sujeira

In [ ]:
def build_sheet_report(sheet_name, df):
    profile = {
        "aba": sheet_name,
        "n_linhas": df.shape[0],
        "n_colunas": df.shape[1],
        "memoria_mb": round(df.memory_usage(deep=True).sum() / (1024**2), 3),
        "lista_colunas": list(df.columns)
    }

    # Gera o detalhamento de tipos
    types_df = pd.DataFrame([
        {"coluna": c, **describe_column_type(df[c])}
        for c in df.columns
    ])

    # Gera o relatório de sujeira
    dirty_df = detect_dirty_patterns(df)

    return profile, types_df, dirty_df

# Exportar resultados para análise gráfica

In [ ]:

def export_analysis_outputs(
    original_sheets,
    final_sheets,
    data_dictionary,
    dirty_reports,
    dataset_summary,
    aggregated_sheets=None,
    output_dir=Path("saida_analise")
):
    output_dir.mkdir(exist_ok=True)

    dataset_summary.to_csv(output_dir / "resumo_dataset.csv", index=False, encoding="utf-8-sig")
    data_dictionary.to_csv(output_dir / "dicionario_de_dados.csv", index=False, encoding="utf-8-sig")

    # um Excel com múltiplas abas de saída
    with pd.ExcelWriter(output_dir / "analise_completa.xlsx", engine="openpyxl") as writer:
        dataset_summary.to_excel(writer, sheet_name="Resumo Dataset", index=False)
        data_dictionary.to_excel(writer, sheet_name="Dicionario de Dados", index=False)

        for sheet_name, report_df in dirty_reports.items():
            safe_name = f"sujeiras_{sheet_name}"[:31]
            report_df.to_excel(writer, sheet_name=safe_name, index=False)

        for sheet_name, df in final_sheets.items():
            safe_name = f"limpo_{sheet_name}"[:31]
            df.to_excel(writer, sheet_name=safe_name, index=False)

        if aggregated_sheets:
            for sheet_name, df in aggregated_sheets.items():
                safe_name = f"agrup_{sheet_name}"[:31]
                df.to_excel(writer, sheet_name=safe_name, index=False)

    print(f"Arquivos exportados em: {output_dir.resolve()}")

export_analysis_outputs(
    original_sheets=sheets,
    final_sheets=final_sheets,
    data_dictionary=data_dictionary,
    dirty_reports=dirty_reports,
    dataset_summary=dataset_summary,
    aggregated_sheets=aggregated_sheets,
    output_dir=OUTPUT_DIR
)

Arquivos exportados em: /content/saida_analise


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
